# Week 7 Exercise — Fine-Tuned Open-Source Product Pricer

**Week 7 capstone:** Fine-tune an **open-source model** (e.g. LLaMA with QLoRA) for the product pricer. Training is done in **Google Colab** (see course Day 1–5 links). This notebook **loads the instructor's pre-fine-tuned model** from HuggingFace and **evaluates** it on the test set.

- **Training:** Run in Colab when you're ready (QLoRA, GPU).
- **This notebook:** Load `ed-donner/pricer-*` + test data → predict → MAE.
- **Requires:** `HF_TOKEN` in `.env`, and preferably a GPU or high RAM for model load.

In [ ]:
import os
import re
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
import torch
from tqdm import tqdm

In [ ]:
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HuggingFace logged in.")
else:
    print("Set HF_TOKEN in .env to load the model and dataset.")

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
FINETUNED_MODEL = "ed-donner/pricer-2024-09-13_13.04.39"
REVISION = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"
DATASET_NAME = "ed-donner/pricer-data"
QUANT_4_BIT = True
EVAL_SIZE = 50

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset["test"]
test_subset = test.select(range(min(EVAL_SIZE, len(test))))
print(f"Test subset: {len(test_subset)} items")

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=QUANT_4_BIT,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

fine_tuned_model = PeftModel.from_pretrained(
    base_model, FINETUNED_MODEL, revision=REVISION
)
fine_tuned_model.eval()
print("Model loaded.")

In [ ]:
def extract_price(s):
    if "Price is $" in s:
        part = s.split("Price is $")[1].replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", part)
        return float(m.group()) if m else 0.0
    return 0.0

def predict(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)
    with torch.no_grad():
        out = fine_tuned_model.generate(
            **inputs, max_new_tokens=8, num_return_sequences=1, pad_token_id=tokenizer.pad_token_id
        )
    return extract_price(tokenizer.decode(out[0]))

In [ ]:
errors = []
for i in tqdm(range(len(test_subset)), desc="Evaluating"):
    row = test_subset[i]
    prompt = row["text"] if "text" in row else row.get("prompt", "")
    truth = float(row["price"])
    pred = predict(prompt)
    errors.append(abs(pred - truth))

mae = sum(errors) / len(errors) if errors else 0
print(f"Test MAE (n={len(errors)}): ${mae:.2f}")